# Stacking untuk Klasifikasi

Course 3: Supervised Learning, Klasifikasi

Stacking menggabungkan beberapa model klasifikasi yang disebut **base learner**, lalu memakai keluaran mereka sebagai masukan bagi satu model lagi yang disebut **meta-classifier** (final estimator). Jadi meta-classifier belajar cara terbaik menggabungkan prediksi para base learner.

Kalau semua model dilatih dan dinilai pada data yang sama, hasilnya mudah overfit. Karena itu `StackingClassifier` memakai cross validation: prediksi base learner untuk meta-classifier dibuat dari data yang tidak dipakai melatih base learner pada putaran itu.

## Dataset

Dua dataset, dibaca langsung dari link:

- `wine.data`: analisis kimia anggur dari tiga jenis (cultivar) di satu wilayah Italia. Targetnya jenis anggur (`Class`: 1, 2, atau 3). Alamat: `https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML241EN-SkillsNetwork/labs/datasets/wine.data`
- `drug200.csv`: pasien dan obat yang cocok, dipakai untuk latihan grid search. Alamat: `https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/drug200.csv`

## 1. Import

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
RS = 42


def akurasi(model, X_train, X_test, y_train, y_test):
    return {"train": round(accuracy_score(y_train, model.predict(X_train)), 4),
            "test": round(accuracy_score(y_test, model.predict(X_test)), 4)}

## 2. Data anggur (wine)

In [ ]:
URL_WINE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML241EN-SkillsNetwork/labs/datasets/wine.data"
kolom = ["Class", "Alcohol", "Malic acid", "Ash", "Alcalinity of ash", "Magnesium", "Total phenols",
         "Flavanoids", "Nonflavanoid phenols", "Proanthocyanins", "Color intensity", "Hue",
         "OD280/OD315 of diluted wines", "Proline"]
wine = pd.read_csv(URL_WINE, names=kolom)
print(wine.shape)
print("kelas:", sorted(wine["Class"].unique().tolist()), "| jumlah per kelas:", wine["Class"].value_counts().sort_index().tolist())
wine.head()

Semua fitur numerik. Berikut korelasi tiap fitur dengan kelas anggur, serta sebaran beberapa fitur yang berkorelasi kuat.

In [ ]:
korelasi = wine.corr()["Class"].drop("Class").sort_values(key=abs, ascending=False)
korelasi.round(2).head(6)

In [ ]:
top3 = korelasi.index[:3].tolist()
sns.pairplot(wine[top3 + ["Class"]], hue="Class", palette="Set1", height=2.2)
plt.show()

Pada tiga fitur teratas, ketiga jenis anggur sudah terlihat cukup terpisah, jadi masalah ini seharusnya relatif mudah bagi model klasifikasi.

## 3. Persiapan data

Scaling dimasukkan ke dalam pipeline, supaya scaler hanya belajar dari data latih (di dalam cross validation juga).

In [ ]:
X = wine.drop(columns="Class")
y = wine["Class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
print("train:", X_train.shape, "| test:", X_test.shape)

## 4. Stacking: SVM + KNN + Decision Tree, meta-classifier Logistic Regression

In [ ]:
base = [("SVM", SVC(random_state=RS)),
        ("KNN", KNeighborsClassifier()),
        ("dt", DecisionTreeClassifier(random_state=RS))]

stack = Pipeline([
    ("scaler", StandardScaler()),
    ("stack", StackingClassifier(estimators=base, final_estimator=LogisticRegression())),
])
stack.fit(X_train, y_train)
print("stacking:", akurasi(stack, X_train, X_test, y_train, y_test))

Sebagai pembanding, akurasi tiap base learner kalau dipakai sendirian:

In [ ]:
sendiri = {}
for nama, model in base:
    m = Pipeline([("scaler", StandardScaler()), ("model", model)]).fit(X_train, y_train)
    sendiri[nama] = akurasi(m, X_train, X_test, y_train, y_test)
sendiri["stacking"] = akurasi(stack, X_train, X_test, y_train, y_test)
pd.DataFrame(sendiri).T

Stacking tidak selalu mengalahkan model terbaik secara individual, apalagi pada data kecil dan mudah seperti ini. Keuntungannya baru terasa saat model-model dasarnya membuat kesalahan yang berbeda-beda. Catatan: seperti model kompleks lainnya, stacking rentan overfit.

### Latihan: tukar peran model

Sekarang base learner-nya SVM, KNN, dan Logistic Regression, sedangkan meta-classifier-nya Decision Tree.

In [ ]:
base2 = [("SVM", SVC(random_state=RS)),
         ("KNN", KNeighborsClassifier()),
         ("lr", LogisticRegression())]

stack2 = Pipeline([
    ("scaler", StandardScaler()),
    ("stack", StackingClassifier(estimators=base2, final_estimator=DecisionTreeClassifier(random_state=RS))),
])
stack2.fit(X_train, y_train)

pd.DataFrame({"meta: Logistic Regression": akurasi(stack, X_train, X_test, y_train, y_test),
              "meta: Decision Tree": akurasi(stack2, X_train, X_test, y_train, y_test)}).T

## 5. GridSearchCV pada stacking (data obat)

In [ ]:
URL_DRUG = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/drug200.csv"
drug = pd.read_csv(URL_DRUG)
print(drug.shape)
drug.head()

Kolom kategorikal (jenis kelamin, tekanan darah, kolesterol) diubah menjadi angka dengan `OrdinalEncoder` di dalam pipeline, lalu semua fitur distandardisasi.

In [ ]:
Xd = drug[["Age", "Sex", "BP", "Cholesterol", "Na_to_K"]]
yd = drug["Drug"]
Xd_train, Xd_test, yd_train, yd_test = train_test_split(Xd, yd, test_size=0.3, random_state=4)

praproses = ColumnTransformer([
    ("kategori", OrdinalEncoder(categories=[["F", "M"], ["LOW", "NORMAL", "HIGH"], ["NORMAL", "HIGH"]]),
     ["Sex", "BP", "Cholesterol"]),
    ("angka", "passthrough", ["Age", "Na_to_K"]),
])

stack_drug = Pipeline([
    ("prep", praproses),
    ("scaler", StandardScaler()),
    ("stack", StackingClassifier(
        estimators=[("SVM", SVC(random_state=RS)), ("knn", KNeighborsClassifier()),
                    ("dt", DecisionTreeClassifier(random_state=RS))],
        final_estimator=LogisticRegression(max_iter=1000))),
])
print("sebelum tuning:", akurasi(stack_drug.fit(Xd_train, yd_train), Xd_train, Xd_test, yd_train, yd_test))

Untuk mengubah hyperparameter base learner, nama parameternya ditulis dengan pola `langkah__namabase__parameter`.

In [ ]:
param_grid = {
    "stack__dt__max_depth": [2, 4, 6, 8],
    "stack__SVM__C": [0.1, 1, 10],
    "stack__SVM__kernel": ["linear", "rbf"],
    "stack__knn__n_neighbors": [3, 5, 9],
}

cari = GridSearchCV(stack_drug, param_grid, scoring="accuracy", cv=5, n_jobs=-1)
cari.fit(Xd_train, yd_train)

print("skor cross validation terbaik:", round(cari.best_score_, 4))
print("parameter terbaik:")
for k, v in cari.best_params_.items():
    print(f"  {k}: {v}")
print("akurasi setelah tuning:", akurasi(cari, Xd_train, Xd_test, yd_train, yd_test))

## Ringkasan

- Stacking memakai prediksi beberapa base learner sebagai masukan meta-classifier.
- Cross validation di dalam `StackingClassifier` mengurangi overfitting yang muncul kalau semua model dilatih di data yang sama.
- Stacking bisa membantu bila base learner membuat kesalahan yang berbeda, tapi tidak selalu lebih baik dari model terbaik individual, dan modelnya lebih sulit dijelaskan.
- Hyperparameter semua base learner bisa dituning bersamaan lewat `GridSearchCV`.